# 00 - Patient Selection
Downloads the numeric files of all ICU stays in MIMIC-IV Waveform Database v0.1.0. and selects 30 patients for our project

## 1. Setup
Load the libraries and define where the data is online (BASE) and where it is locally (DATA_DIR)

In [3]:
import os
import urllib.request
import pandas as pd

BASE = "https://physionet.org/files/mimic4wdb/0.1.0/"
DATA_DIR = os.path.join("..", "dataset", "mimic4wdb")
os.makedirs(DATA_DIR, exist_ok=True)

## 2. List of patients
Read the dataset's main RECORDS file: official list of all 198 patients

In [2]:
patients = pd.read_csv(BASE + "RECORDS", header=None, names=["path"])["path"].tolist()
print(len(patients), "patients")
print(patients[:3])

198 patients
['waves/p100/p10014354/', 'waves/p100/p10019003/', 'waves/p100/p10020306/']


## 3. Link each patient to their ICU stay(s) 
Each patient folder has its own RECORDS file with the stay(s) number(s). We need the stay number because the data file is named after the stay and not the patient. 
Larger pauses prevent server errors (502) and retries handle temporary server errors (502).

In [5]:
import time

def fetch_text(url, tries=5):
    for attempt in range(1, tries + 1):
        try:
            return urllib.request.urlopen(url, timeout=30).read().decode()
        except Exception as e:
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Gave up on {url}")

records = []
for i, p in enumerate(patients, start=1):
    text = fetch_text(BASE + p + "RECORDS")
    for line in text.split():
        records.append(p + line)
    time.sleep(0.2)
    if i % 20 == 0:
        print(f"{i}/{len(patients)} patients checked")

print(len(records), "records")
print(records[:3])

20/198 patients checked
40/198 patients checked
60/198 patients checked
80/198 patients checked
100/198 patients checked
120/198 patients checked
140/198 patients checked
160/198 patients checked
180/198 patients checked
200 records
['waves/p100/p10014354/81739927/81739927', 'waves/p100/p10019003/87033314/87033314', 'waves/p100/p10020306/83404654/83404654']


## 4. Download the numerics files
We download only the **numerics** (`<stay>n.csv.gz`): the vital signs the monitor derives from the waveforms. They match our question (deterioration over minutes to hours) and are much smaller (stay 83404654: 7.2 MB vs. ~143 MB of waveforms). Decisions can be seen in D1 of 'decisions.md'.

The download is resumable (existing files are skipped), safe (`.part` until complete), retries server errors (502), and records stays without numerics (404), which are excluded.

In [7]:
import urllib.error

REDOWNLOAD = False   # change to True to download EVERYTHING again

def download(url, target, tries=5):
    for attempt in range(1, tries + 1):
        try:
            urllib.request.urlretrieve(url, target + ".part")
            os.replace(target + ".part", target)
            return "ok"
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return "missing"
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
        except Exception as e:
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
    return "failed"

skipped, downloaded, missing, failed = [], [], [], []

for i, rec in enumerate(records, start=1):
    parts = rec.split("/")
    patient, record_id = parts[2], parts[-1]
    filename = f"{patient}_{record_id}n.csv.gz"
    target = os.path.join(DATA_DIR, filename)

    if os.path.exists(target) and not REDOWNLOAD:
        skipped.append(rec)
    else:
        result = download(BASE + rec + "n.csv.gz", target)
        if result == "ok":
            downloaded.append(rec)
        elif result == "missing":
            missing.append(rec)
        else:
            failed.append(rec)
        time.sleep(0.2)

    if i % 20 == 0:
        print(f"{i}/{len(records)} records checked")

print()
print(len(skipped), "already there (skipped)")
print(len(downloaded), "newly downloaded")
print(len(missing), "have no numerics file (404)")
print(len(failed), "failed:", failed)

20/200 records checked
40/200 records checked
60/200 records checked
80/200 records checked
100/200 records checked
120/200 records checked
  attempt 1 failed (HTTP Error 502: Bad Gateway), waiting 5 s...
140/200 records checked
160/200 records checked
180/200 records checked
200/200 records checked

56 already there (skipped)
138 newly downloaded
6 have no numerics file (404)
0 failed: []


## 5. Check the download
Count the downloaded numerics files and their total size, as a check.
**Result:** 194 files, 579 MB in total.
- 200 ICU stays in the dataset; 6 have no numerics file (404) and are excluded, leaving 194.
- The numerics are ~4.5% of the full dataset (579 MB vs. 12.8 GB, which is mostly waveforms)

In [7]:
files = [f for f in os.listdir(DATA_DIR) if f.endswith("n.csv.gz")]
total_mb = sum(os.path.getsize(os.path.join(DATA_DIR, f)) for f in files) / 1e6
print(len(files), "files,", round(total_mb), "MB in total")

194 files, 579 MB in total


## 6. Look inside a numerics file
Example: patient p10020306, stay 83404654.

**Findings:**
- The file has 21 columns: `time` plus 20 monitor values.
- We use/focus 5 vital signs: heart rate `HR [bpm]`, systolic blood pressure `NBPs [mmHg]`, diastolic blood pressure `NBPd [mmHg]`, respiratory rate `RR [rpm]`, oxigen saturation `SpO2 [%]` (see D6).
- Each row is a moment where the monitor recorded at least one value, so most cells are empty (`NaN`). The cuff blood pressure is only measured every few minutes.
- `time` is in ticks of ~1 ms (999.56 ticks per second, from the record header). Rows are roughly 1 second apart.
- The rows are **not sorted by time** (e.g. 0 → −4549 → 672), so we must sort them before any time-based step.

In [8]:
example = os.path.join(DATA_DIR, "p10020306_83404654n.csv.gz")
df = pd.read_csv(example)

print(df.shape)
print(list(df.columns))
df.head()

(756823, 21)
['time', 'ABPd [mmHg]', 'ABPm [mmHg]', 'ABPs [mmHg]', 'DeltaQTc [msec]', 'HR [bpm]', 'NBPd [mmHg]', 'NBPm [mmHg]', 'NBPs [mmHg]', 'PVC [/min]', 'Pulse (ABP) [bpm]', 'Pulse (NBP) [bpm]', 'Pulse (SpO2) [bpm]', 'QT [msec]', 'QT-HR [bpm]', 'QTc [msec]', 'RR [rpm]', 'ST-III [mm]', 'ST-V [mm]', 'SpO2 [%]', 'btbHR [bpm]']


,time,ABPd [mmHg],ABPm [mmHg],ABPs [mmHg],DeltaQTc [msec],HR [bpm],NBPd [mmHg],NBPm [mmHg],NBPs [mmHg],PVC [/min],...,Pulse (NBP) [bpm],Pulse (SpO2) [bpm],QT [msec],QT-HR [bpm],QTc [msec],RR [rpm],ST-III [mm],ST-V [mm],SpO2 [%],btbHR [bpm]
0,0,NaN,NaN,NaN,NaN,NaN,77.0,91.0,122.0,NaN,...,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-4549,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.64
2,672,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,19.0,NaN,NaN,NaN,NaN
3,1696,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
4,2720,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21.0,NaN,NaN,NaN,NaN


## 7. How often is each vital sign measured? (one patient)
For patient p10020306 (stay 83404654), we count the measurements of our 5 vital signs, the typical (median) gap between them, and the longest gap.

**Result:**
- HR, RR and SpO2 are measured about **every second** (~265,000 values each); their longest gap is ~24 minutes.
- The cuff blood pressure (NBPs/NBPd) has only ~370 values, and a **25.8-hour gap** without any cuff measurement.
- NBPd has fewer values than NBPs (353 vs. 385): some cuff readings have no diastolic value.

The cuff is the limiting signal, and a long stay is not automatically a complete one. This is why we need a coverage criterion (**Section 9**).

In [9]:
signals = ["HR [bpm]", "RR [rpm]", "SpO2 [%]", "NBPs [mmHg]", "NBPd [mmHg]"]

rows = []
for s in signals:
    t = df.loc[df[s].notna(), "time"].sort_values()
    gaps_s = t.diff() / 999.56
    rows.append({
        "signal": s,
        "measurements": len(t),
        "median gap (s)": round(gaps_s.median(), 1),
        "longest gap (h)": round(gaps_s.max() / 3600, 1),
    })

pd.DataFrame(rows)

,signal,measurements,median gap (s),longest gap (h)
0,HR [bpm],266438,1.0,0.4
1,RR [rpm],264753,1.0,0.4
2,SpO2 [%],264797,1.0,0.4
3,NBPs [mmHg],385,97.3,25.8
4,NBPd [mmHg],353,112.7,25.8


### 7.1 Zoom in: the cuff blood pressure (NBP)

**Result:** mean gap 11.9 min, median 1.6 min, max 1548 min (25.8 h).
- The mean is pulled up by the one 26-hour gap (an outlier), so it is misleading.
- A median of 1.6 min (and a minimum of 5 s) is too short for a real cuff. Possibly the monitor repeats values → see open questions.

In [10]:
nbp = df[["time", "NBPs [mmHg]"]].dropna().sort_values("time")
gaps_min = nbp["time"].diff() / 999.56 / 60
print(len(nbp), "cuff measurements")
print(gaps_min.describe())

385 cuff measurements
count     384.000000
mean       11.915910
std       112.314560
min         0.085371
25%         0.597596
50%         1.622047
75%         3.500207
max      1548.457321
Name: time, dtype: float64


## 8. Duration and coverage of all 194 stays
For each stay we calculate its duration (hours) and, for each of our 5 vital signs (HR, RR, SpO2, NBPs, NBPd), its **coverage**: the percentage of hours that contain at least one measurement.

`min_coverage` is the **lowest** of these 5 coverages. A stay passes a coverage rule only if **every one of the 5 signals** meets it, because our models need all 5 signals together; the weakest signal decides. In practice this is almost always the cuff blood pressure (NBP).

Example: p10020306 has 100% for HR, RR and SpO2 but 23.4% for NBP → min_coverage = 23.4%.

In [11]:
signals = ["HR [bpm]", "RR [rpm]", "SpO2 [%]", "NBPs [mmHg]", "NBPd [mmHg]"]
TICKS_PER_SECOND = 999.56

def stay_summary(path):
    df_s = pd.read_csv(path, usecols=lambda c: c == "time" or c in signals)
    hours = (df_s["time"] - df_s["time"].min()) / TICKS_PER_SECOND / 3600
    hour_block = hours.astype(int)
    n_blocks = hour_block.max() + 1

    row = {
        "file": os.path.basename(path),
        "patient": os.path.basename(path).split("_")[0],
        "duration_h": round(hours.max(), 1),
    }
    for s in signals:
        if s in df_s.columns:
            covered = hour_block[df_s[s].notna()].nunique()
        else:
            covered = 0
        row[s] = round(100 * covered / n_blocks, 1)
    row["min_coverage"] = min(row[s] for s in signals)
    return row

In [12]:
files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith("n.csv.gz"))

summary = []
for i, f in enumerate(files, start=1):
    summary.append(stay_summary(os.path.join(DATA_DIR, f)))
    if i % 20 == 0:
        print(f"{i}/{len(files)} stays checked")

coverage = pd.DataFrame(summary)
coverage.to_csv(os.path.join("..", "dataset", "coverage_summary.csv"), index=False)
print(len(coverage), "stays summarized")
coverage.describe()

20/194 stays checked
40/194 stays checked
60/194 stays checked
80/194 stays checked
100/194 stays checked
120/194 stays checked
140/194 stays checked
160/194 stays checked
180/194 stays checked
194 stays summarized


,duration_h,HR [bpm],RR [rpm],SpO2 [%],NBPs [mmHg],NBPd [mmHg],min_coverage
count,194.000000,194.000000,194.000000,194.000000,194.000000,194.000000,194.000000
mean,46.285567,96.806186,96.900000,95.741753,74.335567,74.335567,70.266495
std,54.684289,16.131929,14.632447,14.829698,34.905967,34.905967,35.915972
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.825000,100.000000,100.000000,100.000000,43.125000,43.125000,37.975000
50%,29.400000,100.000000,100.000000,100.000000,98.350000,98.350000,92.850000
75%,61.225000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
max,342.200000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [13]:
coverage[coverage["patient"] == "p10020306"]

,file,patient,duration_h,HR [bpm],RR [rpm],SpO2 [%],NBPs [mmHg],NBPd [mmHg],min_coverage
2,p10020306_83404654n.csv.gz,p10020306,76.3,100.0,100.0,100.0,23.4,23.4,23.4


**Result (194 stays):**
- Median duration 29.4 h (25%: 12.8 h, max 342 h).
- HR, RR and SpO2 have ~100% coverage in most stays; the cuff (NBP) is the limiting signal: median 98%, but a quarter of the stays have < 43%.
- Sanity check p10020306: 76.3 h, HR/RR/SpO2 100%, but **NBP only 23.4%** → the longest stay is not the most complete one.

In [14]:
for min_hours in [12, 24]:
    for min_cov in [70, 80, 90]:
        ok = (coverage["duration_h"] >= min_hours) & (coverage["min_coverage"] >= min_cov)
        print(f"duration >= {min_hours:>2} h and min_coverage >= {min_cov}%: "
              f"{ok.sum():>3} stays, {coverage.loc[ok, 'patient'].nunique():>3} patients")

duration >= 12 h and min_coverage >= 70%:  97 stays,  97 patients
duration >= 12 h and min_coverage >= 80%:  89 stays,  89 patients
duration >= 12 h and min_coverage >= 90%:  81 stays,  81 patients
duration >= 24 h and min_coverage >= 70%:  75 stays,  75 patients
duration >= 24 h and min_coverage >= 80%:  70 stays,  70 patients
duration >= 24 h and min_coverage >= 90%:  63 stays,  63 patients


**How many stays pass?** Even the strictest combination (≥ 24 h, ≥ 90%) leaves 63 stays (63 patients); the loosest (≥ 12 h, ≥ 70%) leaves 97. Every option exceeds the 30 we planned, so we can use all eligible stays. The final thresholds are decided together (see decision log).

Curiosity check: how many very short stays (less than 1 hour) there are

In [15]:
print((coverage["duration_h"] < 1).sum(), "stays shorter than 1 hour")

8 stays shorter than 1 hour


## 9. How often does the cuff really measure? (input for D7)
The cuff (NBP) is our sparsest signal, so it limits how fine our time grid can be. The monitor re-writes the last displayed cuff value every ~2 minutes, even without a new measurement. We therefore count a cuff row as a **real** measurement only if (NBPs, NBPd, NBPm) differs from the previous cuff row. We use the candidate stays (≥ 12 h, min_coverage ≥ 70%).

In [18]:
candidates = coverage[(coverage["duration_h"] >= 12) & (coverage["min_coverage"] >= 70)]
NBP_COLS = ["NBPs [mmHg]", "NBPd [mmHg]", "NBPm [mmHg]"]

def new_cuff_minutes(d):
    """Minutes since start of each REAL new cuff measurement (copies removed)."""
    minutes = (d["time"] - d["time"].min()) / TICKS_PER_SECOND / 60
    cuff = d.assign(m=minutes).dropna(subset=["NBPs [mmHg]"]).sort_values("time")
    vals = cuff[[c for c in NBP_COLS if c in cuff.columns]].fillna(-1)
    changed = vals.ne(vals.shift()).any(axis=1)
    return minutes, cuff.loc[changed, "m"]

print(len(candidates), "candidate stays")

97 candidate stays


In [19]:
rep = []
for f in candidates["file"]:
    d = pd.read_csv(os.path.join(DATA_DIR, f), usecols=lambda c: c == "time" or c in NBP_COLS)
    minutes, new_minutes = new_cuff_minutes(d)
    cuff_rows = d["NBPs [mmHg]"].notna().sum()
    rep.append({
        "file": f,
        "cuff_rows": cuff_rows,
        "new_values": len(new_minutes),
        "repeats (%)": 100 * (1 - len(new_minutes) / cuff_rows),
        "new_per_hour": len(new_minutes) / (minutes.max() / 60),
    })

repeats = pd.DataFrame(rep)
repeats.describe().round(1)

,cuff_rows,new_values,repeats (%),new_per_hour
count,97.0,97.0,97.0,97.0
mean,1206.6,212.7,83.7,3.7
std,1135.4,244.0,6.4,2.1
min,119.0,9.0,63.9,0.7
25%,486.0,62.0,80.0,2.0
50%,935.0,143.0,85.2,2.9
75%,1422.0,243.0,89.2,4.6
max,6302.0,1289.0,92.4,12.2


**Result (97 stays):** ~85% of cuff rows are copies. Real new cuff measurements occur ~2.9 times per hour (median), i.e. about **every 20 minutes** (25%: every ~30 min, 75%: every ~13 min).
Limitation: two real measurements with exactly the same values count as one (rare).

### 9.1 Which grid size?
For different grid sizes, the percentage of time blocks that contain at least one **real** cuff measurement (copies removed).

In [20]:
grids = [5, 10, 15, 30, 60]

results = []
for f in candidates["file"]:
    d = pd.read_csv(os.path.join(DATA_DIR, f), usecols=lambda c: c == "time" or c in NBP_COLS)
    minutes, new_minutes = new_cuff_minutes(d)
    row = {"file": f}
    for g in grids:
        n_blocks = int(minutes.max() // g) + 1
        row[f"filled_{g}min (%)"] = 100 * (new_minutes // g).nunique() / n_blocks
    results.append(row)

grid_check = pd.DataFrame(results)
grid_check.describe().round(1)

,filled_5min (%),filled_10min (%),filled_15min (%),filled_30min (%),filled_60min (%)
count,97.0,97.0,97.0,97.0,97.0
mean,16.4,30.8,43.5,66.1,92.3
std,9.1,16.5,21.2,19.4,9.8
min,5.3,10.5,13.7,23.1,46.2
25%,9.1,18.1,26.7,50.3,89.5
50%,13.1,24.8,37.7,62.0,96.2
75%,19.6,38.2,55.3,80.7,99.3
max,50.6,100.0,100.0,100.0,100.0


**Result (97 stays, real measurements only, median):** 15-min grid → 38% of blocks contain a real cuff measurement; 30 min → 62%; 60 min → 96%. The cuff is much sparser than the copied values suggested. To discuss for D7.